In [ ]:
# Copyright (c) 2026 Nokia Bell Labs
# Licensed under the BSD 3 Clause license
# SPDX-License-Identifier: BSD-3-Clause

In [ ]:
import re
import ast
import os
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from sklearn.metrics import adjusted_rand_score

In [ ]:
def parse_aggregation_log_old(file_path):
    rounds_data = {}
    aggregation_pattern = re.compile(r"Server: Aggregating model (\d+) based on local models of clients: \[(.*?)\]")
    accuracy_pattern = re.compile(r"For client (\d+) and model (\d+), test accuracy = ([0-9\.]+)")
    round_pattern = re.compile(r"Beginning of round (\d+):")
    
    with open(file_path, 'r') as file:
        current_round = None
        current_mapping = {}
        current_accuracies_1 = {}
        current_accuracies_2 = {}
        recording_second_accuracy = True # Start by looking for train accuracy lines
        
        for line in file:
            round_match = round_pattern.search(line)
            aggregation_match = aggregation_pattern.search(line)
            accuracy_match = accuracy_pattern.search(line)

            # Ignore first test set
            
            if round_match:
                if current_round is not None:
                    rounds_data[current_round] = (current_mapping, current_accuracies_1, current_accuracies_2)
                
                current_round = int(round_match.group(1))
                current_mapping = {}
                current_accuracies_1 = {} # Test accuracy
                current_accuracies_2 = {} # Train accuracy
                recording_second_accuracy = True # Following lines are for training accuracy
            elif aggregation_match:
                model_id = int(aggregation_match.group(1))
                clients = map(int, aggregation_match.group(2).split(', '))
                for client in clients:
                    current_mapping[client] = model_id
                recording_second_accuracy = False # Next we should get lines for test accuracy
            elif accuracy_match:
                client_id = int(accuracy_match.group(1))
                model_id = int(accuracy_match.group(2))
                accuracy = float(accuracy_match.group(3))
                if recording_second_accuracy:
                    if client_id not in current_accuracies_2:
                        current_accuracies_2[client_id] = {}
                    current_accuracies_2[client_id][model_id] = accuracy
                else:
                    if client_id not in current_accuracies_1:
                        current_accuracies_1[client_id] = {}
                    current_accuracies_1[client_id][model_id] = accuracy
            elif line.strip() == "":  # Empty line marks transition to second accuracy set
                transition = True # not used
                #recording_second_accuracy = False # Next we should get lines for test accuracy
        
        if current_round is not None:
            rounds_data[current_round] = (current_mapping, current_accuracies_1, current_accuracies_2)
            print(f"round: {current_round}, clustering: {current_mapping}")
    
    return rounds_data

In [ ]:
def post_process_result_for_CLoVE(rounds_data):
    final_result = {}
    final_result['clusters']={}
    final_result['accuracies']=[]
    for r, data in rounds_data.items():
        final_result['clusters'][r] = [data[0]]

    mean_accuracies = []

    for round_num, (mapping, accuracies_1, accuracies_2) in rounds_data.items():
        acc_list_1 = [accuracies_1[client][mapping[client]] for client in mapping if client in accuracies_1 and mapping[client] in accuracies_1[client]]
        acc_list_2 = [accuracies_2[client][mapping[client]] for client in mapping if client in accuracies_2 and mapping[client] in accuracies_2[client]]
        
        if acc_list_1 and acc_list_2:
            mean_accuracies.append(float(np.mean(acc_list_1)))

    final_result['accuracies'] = mean_accuracies
    return final_result

In [ ]:
def get_rounds_data_for_CLoVE(seed_lines):
    rounds_data = {}
    aggregation_pattern = re.compile(r"Server: Aggregating model (\d+) based on local models of clients: \[(.*?)\]")
    accuracy_pattern = re.compile(r"For client (\d+) and model (\d+), test accuracy = ([0-9\.]+)")
    round_pattern = re.compile(r"Beginning of round (\d+):")

    current_round = None
    current_mapping = {}
    current_accuracies_1 = {}
    current_accuracies_2 = {}
    recording_second_accuracy = True # Start by looking for train accuracy lines
    
    for line in seed_lines:
        round_match = round_pattern.search(line)
        aggregation_match = aggregation_pattern.search(line)
        accuracy_match = accuracy_pattern.search(line)

        # Ignore first test set
        if round_match:
            if current_round is not None:
                rounds_data[current_round] = (current_mapping, current_accuracies_1, current_accuracies_2)
            
            current_round = int(round_match.group(1))
            current_mapping = {}
            current_accuracies_1 = {} # Test accuracy
            current_accuracies_2 = {} # Train accuracy
            recording_second_accuracy = True # Following lines are for training accuracy
        elif aggregation_match:
            model_id = int(aggregation_match.group(1))
            clients = map(int, aggregation_match.group(2).split(', '))
            for client in clients:
                current_mapping[client] = model_id
            recording_second_accuracy = False # Next we should get lines for test accuracy
        elif accuracy_match:
            client_id = int(accuracy_match.group(1))
            model_id = int(accuracy_match.group(2))
            accuracy = float(accuracy_match.group(3))
            if recording_second_accuracy:
                if client_id not in current_accuracies_2:
                    current_accuracies_2[client_id] = {}
                current_accuracies_2[client_id][model_id] = accuracy
            else:
                if client_id not in current_accuracies_1:
                    current_accuracies_1[client_id] = {}
                current_accuracies_1[client_id][model_id] = accuracy
        elif line.strip() == "":  # Empty line marks transition to second accuracy set
            transition = True # not used
            #recording_second_accuracy = False # Next we should get lines for test accuracy
    
    if current_round is not None:
        if current_mapping is None or len(current_mapping) <= 0:
            current_mapping = [i - 1 for i in range(len(current_accuracies_2))]
            print(current_mapping)
        rounds_data[current_round] = (current_mapping, current_accuracies_1, current_accuracies_2)
        print(f"round: {current_round}, clustering: {current_mapping}")

    return rounds_data

In [ ]:
def parse_log_for_CLoVE(file_path):

    line_data = []
    with open(file_path, 'r') as file:

        for line in file:
            line_data.append(line)
        
    return get_rounds_data_for_CLoVE(line_data)

In [ ]:
def process_seed_lines_for_CLoVE(seed_lines, algo='CLoVE'):
    rounds_data = get_rounds_data_for_CLoVE(seed_lines)
    return post_process_result_for_CLoVE(rounds_data)

In [ ]:
def process_seed_lines_for_PACFL(seed_lines, algo='PACFL'):
    data = {
        'clusters': {},
        'accuracies': []
    }

    current_round = None

    for line in seed_lines:
        # Match round line
        round_match = re.match(r"#+ ROUND (\d+) #+", line)
        if round_match:
            current_round = int(round_match.group(1))
            # Ensure accuracy list has enough entries
            while len(data['accuracies']) < current_round:
                data['accuracies'].append(None)
            continue

        # Match clusters line
        if line.startswith("Clusters:"):
            cluster_list_str = line[len("Clusters:"):].strip()
            try:
                cluster_groups = ast.literal_eval(cluster_list_str)
                cluster_dict = {}
                for cluster_id, clients in enumerate(cluster_groups):
                    for client_id in clients:
                        cluster_dict[client_id] = cluster_id
                data['clusters'] = cluster_dict
            except Exception as e:
                print(f"Failed to parse clusters on line: {line}")
            continue

        # Match accuracy line
        acc_match = re.match(r"End round avg\. accuracy: ([\d\.]+)", line)
        if acc_match and current_round is not None:
            accuracy = float(acc_match.group(1)) / 100.0
            data['accuracies'][current_round - 1] = accuracy

    # Assign clusters to all rounds (since PACFL finds one clustering only, in the first round itself)
    clusters = {}
    for i in range(len(data['accuracies'])):
        clusters[i+1] = [data['clusters']]
    data['clusters'] = clusters

    return data

# Example usage:
# result = parse_log_file("your_log_file.txt")
# print(result)

In [ ]:
def extract_cluster_groups_for_CFL(cluster_data_str):
    """
    Extracts a list of lists from a string like:
    '[array([1, 2]), array([3, 4])]' into [[1, 2], [3, 4]]
    """
    array_matches = re.findall(r'array\((\[[^\]]*\])\)', cluster_data_str)
    cluster_groups = []
    for arr in array_matches:
        try:
            clients = ast.literal_eval(arr)
            cluster_groups.append(clients)
        except Exception as e:
            print(f"Failed to parse array: {arr} -> {e}")
    return cluster_groups

In [ ]:
def process_seed_lines_for_CFL(seed_lines, algo='CFL'):
    data = {
        'clusters': {},     # round_num: {client_id: cluster_id}
        'accuracies': []    # index = round_num - 1
    }

    for line in seed_lines:
        line = line.strip()

        # if line.startswith("Starting run with seed="):
        #     break

        # Match clustering line
        cluster_match = re.match(r"For algo=CFL[:,]*\s*clusters[:=]*\s*(\d+),\s*(.*)", line)
        if cluster_match:
            round_num = int(cluster_match.group(1))
            cluster_data_str = cluster_match.group(2)
            try:
                cluster_groups = extract_cluster_groups_for_CFL(cluster_data_str)
                cluster_dict = {}
                for cluster_id, clients in enumerate(cluster_groups):
                    for client_id in clients:
                        cluster_dict[int(client_id)] = cluster_id
                data['clusters'][round_num] = [cluster_dict]  # ✅ Store by round
            except Exception as e:
                print(f"Cluster parse failed for round {round_num}: {e}")
            continue

        # Match accuracy line
        acc_match = re.match(r"For algo=CFL, acc=[:=]? (\d+),\s+(.*)", line)
        if acc_match:
            round_num = int(acc_match.group(1))
            acc_list_str = acc_match.group(2)
            try:
                acc_list = ast.literal_eval(acc_list_str)
                acc_list = [float(a) for a in acc_list]
                avg_acc = float(np.mean(acc_list))
                #print(acc_list, avg_acc)
                while len(data['accuracies']) < round_num:
                    data['accuracies'].append(None)
                data['accuracies'][round_num - 1] = avg_acc
            except Exception as e:
                print(f"Accuracy parse failed for round {round_num}: {e}")
            continue

    return data

In [ ]:

def process_seed_lines_fedgroup(seed_lines, algo='FedGroup'):
    data = {
        'clusters': {},     # {round_num: {client_id: cluster_id}}
        'accuracies': []    # list indexed by round number
    }

    for line in seed_lines:
        line = line.strip()

        # Match the cluster mapping line
        cluster_match = re.match(r"FedGroup: For round (\d+), all_c: (.+)", line)
        if cluster_match:
            round_num = int(cluster_match.group(1))
            cluster_data_str = cluster_match.group(2)
            try:
                cluster_list = ast.literal_eval(cluster_data_str)
                cluster_dict = {}
                for cluster_id, client_list in cluster_list:
                    for client_id in client_list:
                        cluster_dict[int(client_id)] = cluster_id
                data['clusters'][round_num] = [cluster_dict]
            except Exception as e:
                print(f"Failed to parse cluster data in round {round_num}: {e}")
            continue

        # Match the accuracy line (Partial or Complete)
        acc_match = re.match(r"FedGroup: Round (\d+), Test\((Partial|Complete)\) ACC: ([\d.]+)", line)
        if acc_match:
            round_num = int(acc_match.group(1))
            acc = float(acc_match.group(3))

            while len(data['accuracies']) <= round_num:
                data['accuracies'].append(None)
            data['accuracies'][round_num] = acc
            continue

    return data

In [ ]:
def process_seed_lines_FedPAC(seed_lines, algo='FedPAC'):
    data = {
        'clusters': {},     # {round_num: {client_id: cluster_id}}
        'accuracies': []    # list indexed by round number
    }

    for line in seed_lines:
        line = line.strip()

        # Match the cluster mapping line
        cluster_match = re.match(fr"{algo}: For round (\d+), all_c: (.+)", line)
        if cluster_match:
            round_num = int(cluster_match.group(1))
            cluster_data_str = cluster_match.group(2)
            try:
                cluster_list = ast.literal_eval(cluster_data_str)
                cluster_dict = {}
                for cluster_id, client_list in cluster_list:
                    for client_id in client_list:
                        cluster_dict[int(client_id)] = cluster_id
                data['clusters'][round_num] = [cluster_dict]
            except Exception as e:
                print(f"Failed to parse cluster data in round {round_num}: {e}")
            continue

        # Match the accuracy line (Partial or Complete)
        round_match = re.search(r'Round number: (\d+)', line)
        acc_match = re.search(r'Averaged Test Accuracy: ([0-9.]+)', line)

        if round_match:
            round_num = int(round_match.group(1))
        elif acc_match and round_num is not None:
            acc = float(acc_match.group(1))
            while len(data['accuracies']) <= round_num:
                data['accuracies'].append(None)
            data['accuracies'][round_num] = acc

    return data

In [ ]:
def process_seed_lines_pFedMe(seed_lines, algo='pFedMe'):
    data = {
        'clusters': {},     # {round_num: {client_id: cluster_id}}
        'accuracies': []    # list indexed by round number
    }

    for line in seed_lines:
        line = line.strip()

        # Match the cluster mapping line
        cluster_match = re.match(fr"{algo}: For round (\d+), all_c: (.+)", line)
        if cluster_match:
            round_num = int(cluster_match.group(1))
            cluster_data_str = cluster_match.group(2)
            try:
                cluster_list = ast.literal_eval(cluster_data_str)
                cluster_dict = {}
                for cluster_id, client_list in cluster_list:
                    for client_id in client_list:
                        cluster_dict[int(client_id)] = cluster_id
                data['clusters'][round_num] = [cluster_dict]
            except Exception as e:
                print(f"Failed to parse cluster data in round {round_num}: {e}")
            continue

        # Match the accuracy line (Partial or Complete)
        round_match = re.search(r'Round number: (\d+)', line)
        acc_match = re.search(r'Average Test Accuracy: ([0-9.]+)', line)

        if round_match:
            round_num = int(round_match.group(1))
        elif acc_match and round_num is not None:
            acc = float(acc_match.group(1))
            while len(data['accuracies']) <= round_num:
                data['accuracies'].append(None)
            data['accuracies'][round_num] = acc

    return data

In [ ]:
seed_lines = [
    "For algo=CFL:, clusters=: 1,  [array([7, 9, 10]), array([0, 1, 2])]",
    "For algo=CFL, acc=: 1, ['0.995', '0.990', '0.992', '0.981', '0.993', '0.978', '0.984', '0.969', '0.970', '0.972', '0.985', '0.986', '0.972', '0.988', '0.979']",

    "For algo=CFL:, clusters=: 2,  [array([7, 9, 10]), array([0, 1, 2])]",
    "For algo=CFL, acc=: 1, ['0.995', '0.990', '0.992', '0.981', '0.993', '0.978', '0.984', '0.969', '0.970', '0.972', '0.985', '0.986', '0.972', '0.988', '0.979']",
]

In [ ]:
a = process_seed_lines_for_CFL(seed_lines)
print(a)

In [ ]:
def parse_log_file_with_seed_info(file_path, algo='PACFL'):
    seed_data = {}
    current_seed = None
    seed_lines = []
    seeds_found = False

    process_seed_lines = None
    if algo == 'PACFL':
        process_seed_lines = process_seed_lines_for_PACFL
    elif algo =='CFL':
        process_seed_lines = process_seed_lines_for_CFL
    elif algo =='CLoVE' or algo =='IFCA' or algo=='local' or algo=='vanillaFL':
        process_seed_lines = process_seed_lines_for_CLoVE
    elif algo == 'FlexCFL' or algo == 'FeSEM':
        process_seed_lines = process_seed_lines_fedgroup
    elif algo == 'FedPAC' or algo=='PerAvg' or algo=='FedProto':
        process_seed_lines = process_seed_lines_FedPAC
    elif algo=='pFedMe':
        process_seed_lines = process_seed_lines_pFedMe
    else:
        print(f"Algo not supported: {algo}")
        return None

    skip_lines = False  # Flag to skip lines from other algorithms

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()

            # Detect any "Starting run" line
            any_seed_match = re.match(r"Starting run with seed=(\d+),algorithm=([^\s]+)", line)
            if any_seed_match:
                found_seed = int(any_seed_match.group(1))
                found_algo = any_seed_match.group(2)

                if found_algo != algo:
                    # Mismatch: enter skip mode
                    skip_lines = True
                    continue
                else:
                    # Match: end skip mode
                    skip_lines = False
                    seeds_found = True

                    # Process previous seed block
                    if current_seed is not None and seed_lines:
                        seed_data[current_seed] = process_seed_lines(seed_lines, algo)

                    # Start new seed block
                    current_seed = found_seed
                    seed_lines = []
                    continue

            # Accumulate lines only if not skipping
            if not skip_lines:
                seed_lines.append(line)

        # Final block
        if seeds_found:
            if current_seed is not None and seed_lines:
                seed_data[current_seed] = process_seed_lines(seed_lines)
        else:
            # No seeds found — treat whole file as seed 1
            seed_data[1] = process_seed_lines(seed_lines)

    return seed_data

In [ ]:
def parse_log_file_with_seed_info_prev(file_path, algo='PACFL'):
    seed_data = {}
    current_seed = None
    seed_lines = []
    seeds_found = False

    process_seed_lines = None
    if algo == 'PACFL':
        process_seed_lines = process_seed_lines_for_PACFL
    elif algo =='CFL':
        process_seed_lines = process_seed_lines_for_CFL
    elif algo =='CLoVE' or algo == 'IFCA':
        process_seed_lines = process_seed_lines_for_CLoVE
    else:
        print(f"Algo not supported: {algo}")
        return None

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()

            # Detect start of a new seed block
            seed_match = re.match(rf"Starting run with seed=(\d+),algorithm={algo}", line)
            if seed_match:
                seeds_found = True
                # Process previous seed block
                if current_seed is not None:
                    seed_data[current_seed] = process_seed_lines(seed_lines)

                # Start new seed block
                current_seed = int(seed_match.group(1))
                seed_lines = []
                continue

            # Accumulate lines for the current seed
            seed_lines.append(line)

        # Process the last seed block or whole file if no seeds found
        if seeds_found:
            if current_seed is not None:
                seed_data[current_seed] = process_seed_lines(seed_lines)
        else:
            # Assume all data belongs to seed 1
            seed_data[1] = process_seed_lines(seed_lines)

    return seed_data

In [ ]:
def compute_and_plot_stats(rounds_data):
    round_numbers = []
    mean_accuracies_1 = []
    mean_accuracies_2 = []
    
    for round_num, (mapping, accuracies_1, accuracies_2) in rounds_data.items():
        acc_list_1 = [accuracies_1[client][mapping[client]] for client in mapping if client in accuracies_1 and mapping[client] in accuracies_1[client]]
        acc_list_2 = [accuracies_2[client][mapping[client]] for client in mapping if client in accuracies_2 and mapping[client] in accuracies_2[client]]
        
        if acc_list_1 and acc_list_2:
            round_numbers.append(round_num)
            mean_accuracies_1.append(np.mean(acc_list_1))
            mean_accuracies_2.append(np.mean(acc_list_2))
            #print(f"round ={round_num}: ")
            #print(f"acc1 ={acc_list_1}: ")
            #print(f"acc2 ={acc_list_2}: ")
    
    plt.figure(figsize=(10, 5))
    plt.plot(round_numbers, mean_accuracies_1, label='Test accuracy', marker='o')
    plt.plot(round_numbers, mean_accuracies_2, label='Train accuracy', marker='s')
    plt.xlabel("Round Number")
    plt.ylabel("Mean Test Accuracy")
    plt.title("Mean Test Accuracy Over Rounds")
    plt.legend()
    plt.grid(True)
    plt.show()
    x=[round(float(num), 3) for num in mean_accuracies_1]
    print(f"test loss: {x}")


In [ ]:
def find_stabilization_round(rounds_data):
    final_mapping = list(rounds_data.values())[-1][0]  # Get the final client-to-model mapping
    stabilization_rounds = []
    
    for round_num, (mapping, _, _) in rounds_data.items():
        if mapping == final_mapping:
            stabilization_rounds.append(round_num)
    
    # plt.figure(figsize=(10, 5))
    # plt.hist(stabilization_rounds, bins=10, alpha=0.75, color='b', edgecolor='black')
    # plt.xlabel("Round Number")
    # plt.ylabel("Frequency of Stabilization")
    # plt.title("Rounds Where Client-to-Model Mapping Stabilizes")
    # plt.grid(True)
    # plt.show()
    
    return stabilization_rounds

In [ ]:
def plot_stabilization(round_data):
    """
    Plots whether the client-to-model mapping has stabilized at each round.
    
    :param round_data: Dictionary of rounds mapping to (client_mapping, test_acc_1, test_acc_2).
    """
    final_round = max(round_data.keys())  # Get the last round number
    final_mapping = round_data[final_round][0]  # Extract the final mapping

    rounds = sorted(round_data.keys())  # Ensure rounds are in order
    stabilization_status = [1 if round_data[r][0] == final_mapping else 0 for r in rounds]  # 1 if stabilized, else 0

    # Plot stabilization
    plt.figure(figsize=(8, 5))
    plt.plot(rounds, stabilization_status, marker='o', linestyle='-', color='b', label="Stabilization Status")

    plt.xlabel("Round Number")
    plt.ylabel("Stabilized (1) / Not Stabilized (0)")
    plt.title("Client-to-Model Mapping Stabilization Over Rounds")
    plt.yticks([0, 1], labels=["Not Stabilized", "Stabilized"])
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.show()

    return stabilization_status

In [ ]:
def compute_ari(list_labels, dict_labels):
    if len(list_labels) != len(dict_labels):
        return None

    # Convert dict to list using index order
    dict_label_list = [dict_labels[i] for i in range(len(list_labels))]

    # Compute ARI
    ari_score = adjusted_rand_score(list_labels, dict_label_list)
    return ari_score

In [ ]:
def plot_ARI(round_data, ground_truth, seed, algo):
    """
    Plots ARI at each round.
    
    :param round_data: Dictionary of rounds mapping to (client_mapping, test_acc_1, test_acc_2).
    """
    rounds = sorted(round_data.keys())  # Ensure rounds are in order
    ARI_list = [compute_ari(ground_truth, round_data[r][0] ) for r in rounds]  # 1 if stabilized, else 0

    # Plot stabilization
    plt.figure(figsize=(8, 5))
    plt.plot(rounds, ARI_list, marker='o', linestyle='-', color='b', label="Adj. Rand Index")

    plt.xlabel("Round Number")
    #plt.ylabel("Stabilized (1) / Not Stabilized (0)")
    plt.title(f"Clustering Accuracy: seed={seed}, algo={algo}")
    plt.ylim(-0.5, 1.05)
    #plt.yticks([0, 1], labels=["Not Stabilized", "Stabilized"])
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.show()

    return ARI_list

In [ ]:
def compute_mapping_accuracy(round_data):
    """
    Computes the accuracy of the final client-to-model mapping based on adjacency and size consistency.
    Prints accuracy and all valid sets (model: [clients]) in a single line.

    :param round_data: Dictionary of rounds mapping to (client_mapping, test_acc_1, test_acc_2).
    :return: Accuracy score (float).
    """
    final_round = max(round_data.keys())  # Get the last round number
    final_mapping = round_data[final_round][0]  # Extract the final client-to-model mapping
    
    # Group clients by their assigned model
    model_to_clients = defaultdict(list)
    for client, model in final_mapping.items():
        model_to_clients[model].append(client)
    
    # Sort clients within each model
    for model in model_to_clients:
        model_to_clients[model].sort()

    # Step 2: Check adjacency condition
    adjacent_models = set()
    for model, clients in model_to_clients.items():
        if all(clients[i] + 1 == clients[i + 1] for i in range(len(clients) - 1)):  # Clients must be consecutive
            adjacent_models.add(model)

    # Step 3: Count models with the same number of clients
    size_to_models = defaultdict(list)  # Map client group sizes to list of models with that size
    for model in adjacent_models:
        size_to_models[len(model_to_clients[model])].append(model)

    # Step 4: Find the largest set of models with the same number of clients
    max_valid_set_size = max(map(len, size_to_models.values()), default=0)  # Default to 0 if no valid models
    largest_valid_sets = [models for models in size_to_models.values() if len(models) == max_valid_set_size]

    # Step 5: Compute accuracy
    total_models = len(model_to_clients)
    accuracy = max_valid_set_size / total_models if total_models > 0 else 0

    # Print results with client lists
    valid_sets_str = " | ".join([f"{model}:{model_to_clients[model]}" for models in largest_valid_sets for model in models])
    print(f"Accuracy: {accuracy:.3f} | Valid Sets: {valid_sets_str}")

    return accuracy

In [ ]:
def extract_ground_truth_list(run_id, base_log_dir):
    pattern = r"Ground Truth Client to Cluster:\s*\[([0-9,\s]+)\]"

    log_dir = os.path.join(base_log_dir, run_id)
    file_name = run_id +"_pfl_experiments.log"
    file_path = os.path.join(log_dir, file_name)

    with open(file_path, 'r') as file:
        for line in file:
            match = re.search(pattern, line)
            if match:
                number_str = match.group(1)
                number_list = [int(num.strip()) for num in number_str.split(',')]
                return number_list

    print("Line not found or could not extract list.")
    return None

In [ ]:
def get_Algo_results(run_id, base_log_dir, algo='PACFL'):
    log_dir = os.path.join(base_log_dir, run_id)
    file_name = run_id +"_pfl_experiments.log"
    file_path = os.path.join(log_dir, file_name)

    seeds_rounds_info = parse_log_file_with_seed_info(file_path, algo)
    return seeds_rounds_info

In [ ]:
def plot_accuracy_for_run_id(run_id, base_log_dir):
    log_dir = os.path.join(base_log_dir, run_id)
    file_name = run_id +"_pfl_experiments.log"
    file_path = os.path.join(log_dir, file_name)

    rounds_info = parse_log_for_CLoVE(file_path)
    compute_and_plot_stats(rounds_info)

In [ ]:
def plot_clustering_for_run_id(run_id, base_log_dir):
    log_dir = os.path.join(base_log_dir, run_id)
    file_name = run_id +"_pfl_experiments.log"
    file_path = os.path.join(log_dir, file_name)

    rounds_info = parse_log_for_CLoVE(file_path)
    stabilization_achieved = plot_stabilization(rounds_info)
    print("Stabilization per round:", stabilization_achieved)

In [ ]:
def get_clustering_accuracy_for_run_id(run_id, base_log_dir):
    log_dir = os.path.join(base_log_dir, run_id)
    file_name = run_id +"_pfl_experiments.log"
    file_path = os.path.join(log_dir, file_name)

    rounds_info = parse_log_for_CLoVE(file_path)
    accuracy_achieved = compute_mapping_accuracy(rounds_info)
    print(f"Clustering accuracy for {run_id} is: {accuracy_achieved}")

In [ ]:
def plot_ARI_for_run_id(run_id, base_log_dir, algo='PACFL'):
    result = get_Algo_results(run_id, base_log_dir, algo=algo)
    print(result)
    ground_truth = extract_ground_truth_list(run_id, base_log_dir)
    print(f"Ground Truth: {ground_truth}")
    for seed, result_for_seed in result.items():
        ARI_list = plot_ARI(result_for_seed['clusters'], ground_truth, seed, algo)
        print(f"Clustering ARIs for {run_id}:{seed} is: {ARI_list}")

In [ ]:
def update_running_average(running_avg, new_list, count):
    """
    Update or initialize the running average given the current average list,
    a new list, and the number of previous lists seen.

    If running_avg is empty (i.e., first list), just return the new list as float values.
    """
    if not running_avg:
        return [float(x) for x in new_list]

    return [
        (r_avg * count + new_val) / (count + 1)
        for r_avg, new_val in zip(running_avg, new_list)
    ]

In [ ]:
def get_averaged_results(result, ground_truth):
    final_results = {
        "ARI": [],
        "accuracies": []
    }
    count = 0
    for seed, result_for_seed in result.items():
        round_cluster_data = result_for_seed['clusters']
        rounds = sorted(round_cluster_data.keys())  # Ensure rounds are in order
        ARI_list = [compute_ari(ground_truth, round_cluster_data[r][0] ) for r in rounds]  # 1 if stabilized, else 0
        acc_list = result_for_seed['accuracies']
        final_results['ARI'] = update_running_average(final_results['ARI'], ARI_list, count)
        final_results['accuracies'] = update_running_average(final_results['accuracies'], acc_list, count)
        count += 1

    return final_results

In [ ]:
def get_seed_averaged_results_for_algo(run_id, base_log_dir, algo='PACFL'):
    result = get_Algo_results(run_id, base_log_dir, algo=algo)
    print(result)
    ground_truth = extract_ground_truth_list(run_id, base_log_dir)
    print(f"Ground Truth: {ground_truth}")
    ret_val = get_averaged_results(result, ground_truth)
    return ret_val

In [ ]:
directory_path = "../"
results_dir='results'
base_log_dir = os.path.join(directory_path, results_dir)

In [ ]:
run_id = "423"
algo='PACFL'
run_id='413'
algo='CFL'
algo='CLoVE'
algo='IFCA'
algo='FedPAC'
algo='PerAvg'
algo='FedProto'
algo='pFedMe'
algo='local'
run_id = "414"
#algo='FlexCFL'
#algo='FeSEM'
run_id = "424"
run_id="426"
run_id='1000'
run_id = '1002'
run_id = '1003'
run_id='1004'
avg_results = get_seed_averaged_results_for_algo(run_id, base_log_dir, algo)
print(avg_results)


In [ ]:
run_id = "424"
algo='PACFL'
#run_id='413'
algo='CFL'
algo='CLoVE'
algo='FlexCFL'
algo='FeSEM'
plot_ARI_for_run_id(run_id, base_log_dir, algo)

In [ ]:
run_id = "414"
algo='FlexCFL'
#algo='FeSEM'

plot_ARI_for_run_id(run_id, base_log_dir, algo)

In [ ]:
run_id = "423"
get_clustering_accuracy_for_run_id(run_id, base_log_dir)

In [ ]:
run_id = "416"
plot_accuracy_for_run_id(run_id, base_log_dir)
plot_clustering_for_run_id(run_id, base_log_dir)

In [ ]:
run_id = "111"
plot_accuracy_for_run_id(run_id, base_log_dir)

In [ ]:
run_id = "120"
plot_accuracy_for_run_id(run_id, base_log_dir)

In [ ]:
run_id = "121"
plot_accuracy_for_run_id(run_id, base_log_dir)

In [ ]:
run_id = "122"
plot_accuracy_for_run_id(run_id, base_log_dir)

In [ ]:
run_id = "130"
plot_accuracy_for_run_id(run_id, base_log_dir)

In [ ]:
run_id = "131"
plot_accuracy_for_run_id(run_id, base_log_dir)

**Below is evaluations results for CIFAR-10**

In [ ]:
for i in range(9):
    run_id = str(i+1)
    get_clustering_accuracy_for_run_id(run_id, base_log_dir)

    if i+1 >= 4:
        run_id = str(10+i+1)
        get_clustering_accuracy_for_run_id(run_id, base_log_dir)

    if i+1 >= 4 and i+1 != 8:
        run_id = str(20+i+1)
        get_clustering_accuracy_for_run_id(run_id, base_log_dir)

In [ ]:
for i in range(9):
    run_id = str(i+1)
    print(f"Exp: {run_id}: **********************************************************************************")
    plot_accuracy_for_run_id(run_id, base_log_dir)

    if i+1 >= 4:
        run_id = str(10+i+1)
        print(f"Exp: {run_id}: *With swap")
        plot_accuracy_for_run_id(run_id, base_log_dir)

    if i+1 >= 4 and i+1 != 8:
        run_id = str(20+i+1)
        print(f"Exp: {run_id}: With overlap")
        plot_accuracy_for_run_id(run_id, base_log_dir)

    print("\n\n") 

In [ ]:
for i in range(9):
    run_id = str(i+1)
    print(f"Exp: {run_id}: **********************************************************************************")
    plot_clustering_for_run_id(run_id, base_log_dir)

    if i+1 >= 4:
        run_id = str(10+i+1)
        print(f"Exp: {run_id}: *With swap")
        plot_clustering_for_run_id(run_id, base_log_dir)

    if i+1 >= 4 and i+1 != 8:
        run_id = str(20+i+1)
        print(f"Exp: {run_id}: With overlap")
        plot_clustering_for_run_id(run_id, base_log_dir)

    print("\n\n") 